In [2]:
!git clone https://github.com/ultralytics/yolov5 /content/yolov5
%cd /content/yolov5
!pip install -r requirements.txt -q
%cd /content

Cloning into '/content/yolov5'...
remote: Enumerating objects: 17898, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 17898 (delta 41), reused 14 (delta 13), pack-reused 17837 (from 5)
Receiving objects: 100% (17898/17898), 17.03 MiB | 15.31 MiB/s, done.
Resolving deltas: 100% (12179/12179), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 10.4 MB/s eta 0:00:00
/content
✅ Ready


In [8]:
!pip install torch torchvision tqdm -q

import os, zipfile, urllib.request, time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.optim.lr_scheduler import OneCycleLR

In [9]:
os.makedirs('/content/data', exist_ok=True)

print("Downloading GTSRB...")
urllib.request.urlretrieve(
    "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip",
    "/content/data/train.zip"
)
urllib.request.urlretrieve(
    "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip",
    "/content/data/test.zip"
)
urllib.request.urlretrieve(
    "https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip",
    "/content/data/test_gt.zip"
)

print("Extracting...")
for f in ['train.zip', 'test.zip', 'test_gt.zip']:
    with zipfile.ZipFile(f'/content/data/{f}') as z:
        z.extractall('/content/data/')
print("Done.")

Extracting...
Done.


In [10]:
import csv, shutil
from PIL import Image

# Parse the ground truth CSV and copy images into class folders
test_img_dir = '/content/data/GTSRB/Final_Test/Images'
test_out_dir = '/content/data/test_sorted'
gt_file = '/content/data/GT-final_test.csv'

with open(gt_file) as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        cls = row['ClassId'].zfill(5)
        src = os.path.join(test_img_dir, row['Filename'])
        dst_dir = os.path.join(test_out_dir, cls)
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(src, dst_dir)

print("Test set organised.")

Test set organised.


In [11]:
BATCH = 128
IMG_SIZE = 48

train_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder('/content/data/GTSRB/Final_Training/Images', transform=train_tfm)
val_ds   = datasets.ImageFolder('/content/data/test_sorted',                  transform=val_tfm)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Classes: {len(train_ds.classes)}")

Train: 39209 | Val: 12630 | Classes: 43


In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

model = models.mobilenet_v2(weights='IMAGENET1K_V1')
model.classifier[1] = nn.Linear(model.last_channel, 43)
model = model.to(device)

Device: cpu
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 99.6MB/s]


In [ ]:
EPOCHS = 15
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = OneCycleLR(optimizer, max_lr=1e-3,
                       steps_per_epoch=len(train_loader), epochs=EPOCHS)

best_acc = 0.0
for epoch in range(1, EPOCHS+1):
    # --- Train ---
    model.train()
    correct = total = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    train_acc = correct / total * 100

    # --- Val ---
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs)
            correct += (out.argmax(1) == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total * 100

    print(f"Epoch {epoch:2d}/{EPOCHS}  Train: {train_acc:.1f}%  Val: {val_acc:.1f}%")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/content/gtsrb_mobilenet.pth')
        print(f"  ✓ Saved (best so far: {best_acc:.1f}%)")

print(f"\nFinal best val accuracy: {best_acc:.1f}%")

In [ ]:
from google.colab import drive
drive.mount('/drive')
shutil.copy('/content/gtsrb_mobilenet.pth', '/drive/MyDrive/gtsrb_mobilenet.pth')
print("Saved to Google Drive.")